**PARTE I: Obtención de métricas**

In [ ]:
import influxdb_client
from influxdb_client.client.write_api import SYNCHRONOUS
from influxdb_client.client.write_api import ASYNCHRONOUS, WriteOptions
from influxdb_client.client.exceptions import InfluxDBError
from urllib3.exceptions import NewConnectionError
from influxdb_client import Point

INFLUX_URL = "http://influxdb2:8086"
INFLUX_TOKEN = "MyInitialAdminToken0="

print("---Iniciando conexión a InfluxDB---")

client = None
try:
    client = influxdb_client.InfluxDBClient(
        url=INFLUX_URL,
        token=INFLUX_TOKEN,
        org="docs"
    )
    write_options = WriteOptions(
        batch_size=500,
        flush_interval=1000,
        write_type=ASYNCHRONOUS
    )
    write_api = client.write_api(write_options=write_options)

    print(f"Verificando estado de salud de InfluxDB en {INFLUX_URL}...")
    health = client.health()
    
    if health.status == "pass":
        print("Conexión exitosa!")
        print(f"Versión del servidor: {health.version}")
    else:
        print(f"ERROR. Conexión fallida. Estado: {health.status}")
        print(f"Mensaje: {health.message}")

except (InfluxDBError, NewConnectionError) as e:
    print("ERROR. Error al conectar con InfluxDB:")
    print(f"   Detalle: {e}")

**Definimos una función para obtener las métricas**

In [ ]:
import psutil
import time

#Obtener estadísticas de uso
def obtener_metricas_sistema(host_id):
    cpu_usage = psutil.cpu_percent(interval=1)
    
    mem = psutil.virtual_memory()
    ram_used_gb = round(mem.used / (1024**3), 2)
    ram_percent = mem.percent
    
    #Uso de disco
    disk = psutil.disk_usage('/')
    disk_percent = disk.percent
    
    return {
        'host': host_id,
        'cpu_percent': cpu_usage,
        'ram_used_gb': ram_used_gb,
        'ram_percent': ram_percent,
        'disk_percent': disk_percent
    }

**Definimos la función que enviará los datos**

In [ ]:
def bucle_lectura():
    while True:
        datos = obtener_metricas_sistema("servidor_A")
        p = (
            Point("rendimiento_servidor")
            .tag("host_id", datos["host"])
            .tag("entorno", "produccion")
            .field("cpu_percent", float(datos["cpu_percent"]))
            .field("ram_percent", float(datos["ram_percent"]))
            .field("disk_percent", float(datos["disk_percent"]))
        )
        write_api.write(bucket="_monitoring", record=p)
        time.sleep(1)
        print("Leyendo datos")

**Ejecutamos el bucle**

In [ ]:
try:
    bucle_lectura()
except KeyboardInterrupt:
    print("Cerrando...")